# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyeshaShahid-creator/Internship-flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content (content_hash_id), for one client (client_hash_id), on one report date (report_date) — a single day's Search Console performance for that content item. I'm developing on month=2026-03 (mid-panel); month=2026-06 (the _sample) is the sealed test month, never touched for label logic.

In [9]:
# Setup — skip if already done earlier in this notebook
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"
PRIOR_MONTH = "2026-02"
FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"

# Schema check — read the printed columns, then fix the constants below
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 1").df())

# TODO: set these to the REAL column names from the output above
AVAILABILITY_COL = "gsc_data_available"
POSITION_COL = "gsc_avg_position"
CLICKS_COL = "gsc_clicks"
IMPRESSIONS_COL = "gsc_impressions"
CLIENT_COL = "client_hash_id"
CONTENT_COL = "content_hash_id"

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: avg daily impressions, avg CTR, avg position, impression volatility, days with data — all computed from March (month=2026-03) only.
Label: is_declining (a proxy, not ground truth) — March average daily clicks lower than February average daily clicks, per (client, content) pair.
Context: client_hash_id, content_hash_id — identifiers used to join and group, not fed into the model as inputs.
Excluded: any metric from April onward, and the sealed month=2026-06 — those exist only to build/validate labels later, never to describe what was known as of the March decision point.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
grain_check = con.sql(f"""
    SELECT {CLIENT_COL}, {CONTENT_COL}, report_date, COUNT(*) AS n
    FROM read_parquet('{FACT}')
    WHERE month = '{MID_MONTH}'
    GROUP BY {CLIENT_COL}, {CONTENT_COL}, report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print(f"Duplicate combos: {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate combos: 0


,client_hash_id,content_hash_id,report_date,n


In [11]:
shape = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
        COUNT(DISTINCT {CLIENT_COL}) AS n_clients, COUNT(DISTINCT {CONTENT_COL}) AS n_content
    FROM read_parquet('{FACT}')
    WHERE month = '{MID_MONTH}'
""").df()
shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [12]:
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE {AVAILABILITY_COL} IS TRUE) AS available_rows
    FROM read_parquet('{FACT}')
    WHERE month = '{MID_MONTH}'
""").df()
avail['pct_available'] = (avail['available_rows'] / avail['total_rows'] * 100).round(1)
avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,3611061,36.7


In [13]:
features = con.sql(f"""
    SELECT
        {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
        AVG({IMPRESSIONS_COL})                        AS avg_daily_impressions,
        SUM({CLICKS_COL}) * 1.0 / NULLIF(SUM({IMPRESSIONS_COL}), 0) AS avg_ctr,
        AVG({POSITION_COL})                           AS avg_position,
        STDDEV({IMPRESSIONS_COL})                      AS impression_volatility,
        COUNT(*)                                       AS days_with_data
    FROM read_parquet('{FACT}')
    WHERE month = '{MID_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()
features

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_id,content_id,avg_daily_impressions,avg_ctr,avg_position,impression_volatility,days_with_data
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,210.419355,0.001073,7.209549,120.140688,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,14.612903,0.000000,2.987198,4.160748,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,181.612903,0.001066,6.724039,82.975771,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,159.483871,0.002629,7.244844,70.257086,31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,13.838710,0.002331,4.209227,5.899699,31
...,...,...,...,...,...,...,...
176733,client_2b4306c3ed003f01,content_4a46b45b40a57f11,1.000000,0.000000,47.000000,NaN,1
176734,client_2b4306c3ed003f01,content_9eeeb3f77716113c,1.000000,0.000000,9.000000,NaN,1
176735,client_20259bd6705d81d4,content_2d8e52b436a736c8,3.000000,0.000000,7.333333,NaN,1
176736,client_20259bd6705d81d4,content_4d0dafdb2450a480,15.000000,0.000000,42.466667,NaN,1


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd

prior = con.sql(f"""
    SELECT {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id, AVG({CLICKS_COL}) AS feb_avg_clicks
    FROM read_parquet('{FACT}')
    WHERE month = '{PRIOR_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

current = con.sql(f"""
    SELECT {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id, AVG({CLICKS_COL}) AS mar_avg_clicks
    FROM read_parquet('{FACT}')
    WHERE month = '{MID_MONTH}' AND {AVAILABILITY_COL} IS TRUE
    GROUP BY {CLIENT_COL}, {CONTENT_COL}
""").df()

trend = prior.merge(current, on=['client_id', 'content_id'])
trend['is_declining'] = (trend['mar_avg_clicks'] < trend['feb_avg_clicks']).astype(int)
trend['clicks_change_pct'] = (trend['mar_avg_clicks'] - trend['feb_avg_clicks']) / trend['feb_avg_clicks'].replace(0, pd.NA)

df = features.merge(trend[['client_id', 'content_id', 'is_declining', 'clicks_change_pct']],
                     on=['client_id', 'content_id']).dropna()

honest_cols = ['avg_daily_impressions', 'avg_ctr', 'avg_position', 'impression_volatility', 'days_with_data']
leak_cols = honest_cols + ['clicks_change_pct']

def quick_auc(cols):
    X, y = df[cols], df['is_declining']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])

print('WITH the leak column:', round(quick_auc(leak_cols), 3), ' <- suspiciously close to 1.0')
print('HONEST (5 features, no leak):', round(quick_auc(honest_cols), 3), ' <- the real number to report')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WITH the leak column: 1.0  <- suspiciously close to 1.0
HONEST (5 features, no leak): 0.692  <- the real number to report


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice only includes the 36.7% of March rows where gsc_data_available is true — the unbalanced panel means some clients/content have partial or no GSC history for parts of the month. The feature frame also only keeps (client, content) pairs present in both February and March (via inner join for the trend label), silently dropping newly published or recently discontinued content — a survivorship gap in exactly the content most likely to need attention.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.